In [1]:
import sys
sys.path.append("../src")

from juris_summarizer import get_client, chunk_text, summarize_chunk, run_batch_eval
from juris_summarizer.prompts import format_chunk_summaries

In [2]:
map_client = get_client("groq")
reduce_client = get_client("cerebras")

In [ ]:
import pandas as pd 
import tiktoken

enc = tiktoken.get_encoding("cl100k_base")

df = pd.read_csv("../data/train.csv")
df['text_words'] = df['text'].apply(lambda x: len(str(x).split()))
df['summary_words'] = df['summary'].str.split().apply(len)
df['compression_ratio'] = df['summary_words'] / df['text_words']
df['text_tokens'] = df['text'].apply(lambda x: len(enc.encode(x)))

In [4]:
df['cr_decile'] = pd.qcut(df['compression_ratio'], 10, labels=False)
sample = df.groupby('cr_decile', group_keys=False).apply(lambda g: g.sample(1, random_state=42))

In [ ]:
# few-shot example — map step uses map_client
example_idx = 713
example_reference_summary = df.loc[example_idx, "summary"]
example_chunks = chunk_text(df.loc[example_idx, "text"])

example_summaries = [
    summarize_chunk(map_client, c, i + 1, len(example_chunks))
    for i, c in enumerate(example_chunks)
]
example_chunk_summaries = format_chunk_summaries(example_summaries)

# batch eval — pass both clients through
results_df = run_batch_eval(
    map_client, reduce_client, sample, example_chunk_summaries, example_reference_summary
)